# 77: Signal Timing Analysis - Are We Missing the Tops?

**Critical Question:** Do Check framework signals cause us to MISS market tops and hold through the most profitable times?

## Hypothesis:

The user suspects:
- ❌ Exit signals trigger TOO LATE (after the dump)
- ❌ Exit signals trigger TOO EARLY (miss the explosive move)
- ❌ Stay in cash during the most profitable periods
- ❌ Re-entry signals are weak or non-existent

## What We'll Investigate:

1. **When do exit signals fire?** (relative to actual tops)
2. **How long are we OUT of the market?** (days in cash)
3. **What price moves do we miss?** (while sitting in cash)
4. **When do re-entry signals trigger?** (how late?)
5. **Trade-by-trade breakdown** (entry price, exit price, missed opportunity)

## Expected Finding:

If the framework is flawed, we'll see:
- Exit at $60k, BTC goes to $100k, re-enter at $90k → MISS THE ENTIRE MOVE
- Or: Exit at $100k (top), but BTC only drops to $95k before rallying to $120k

**This could explain why we underperform buy-and-hold by 87-166%**

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

In [ ]:
# Load data
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')
df = df[df.index >= '2023-01-01'].copy()

print(f"Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
df.head()

## 1. Generate Signals and Run Backtest

In [ ]:
# Entry: Buy The Dip (4/5)
c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0
c5 = (df['liq_long'] / df['liq_short']) > 1.0
entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = entry_count >= 4

# Exit: LTH Distribution
exits = (df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)

# Backtest
pf = vbt.Portfolio.from_signals(
    close=df['price'],
    entries=entries,
    exits=exits,
    fees=0.001,
    slippage=0.001,
    init_cash=10000,
    freq='1D'
)

print(f"Total Return: {pf.total_return() * 100:.1f}%")
print(f"Trades: {pf.trades.count()}")
print(f"Win Rate: {pf.trades.win_rate() * 100:.1f}%")

## 2. Identify Actual Market Tops (Rolling Highs)

In [ ]:
# Find local tops (30-day rolling max)
rolling_max = df['price'].rolling(30, center=True).max()
local_tops = df['price'] == rolling_max

# Find significant tops (price > previous 60-day max)
prev_max = df['price'].rolling(60).max().shift(1)
significant_tops = local_tops & (df['price'] > prev_max)

# Get top dates and prices
top_dates = df.index[significant_tops]
top_prices = df.loc[top_dates, 'price']

print(f"\nSignificant Local Tops (last 3 years):")
print("="*60)
for date, price in zip(top_dates[-10:], top_prices[-10:]):
    print(f"  {date.date()}: ${price:,.0f}")

## 3. Analyze Each Trade: Did We Miss the Top?

In [ ]:
if pf.trades.count() > 0:
    trades = pf.trades.records_readable
    
    print("\n" + "="*100)
    print("TRADE-BY-TRADE ANALYSIS: Did We Miss the Tops?")
    print("="*100)
    
    for i, trade in trades.iterrows():
        entry_date = trade['Entry Date']
        exit_date = trade['Exit Date']
        entry_price = trade['Entry Price']
        exit_price = trade['Exit Price']
        pnl = trade['Return']
        
        # What was the highest price AFTER we exited?
        post_exit_df = df[df.index > exit_date]
        if len(post_exit_df) > 0:
            # Check next 90 days
            next_90_days = post_exit_df.head(90)
            max_price_after = next_90_days['price'].max()
            max_date_after = next_90_days['price'].idxmax()
            missed_gain = (max_price_after / exit_price - 1) * 100
        else:
            max_price_after = exit_price
            max_date_after = exit_date
            missed_gain = 0
        
        # What was the highest price DURING the trade?
        trade_period = df[(df.index >= entry_date) & (df.index <= exit_date)]
        max_during = trade_period['price'].max()
        max_date_during = trade_period['price'].idxmax()
        
        print(f"\nTrade #{i+1}:")
        print(f"  Entry:  {entry_date.date()} @ ${entry_price:,.0f}")
        print(f"  Exit:   {exit_date.date()} @ ${exit_price:,.0f}")
        print(f"  Return: {pnl*100:+.1f}%")
        print(f"  ---")
        print(f"  Peak DURING trade: ${max_during:,.0f} on {max_date_during.date()}")
        print(f"  Peak AFTER exit (90d): ${max_price_after:,.0f} on {max_date_after.date()}")
        
        if max_during > exit_price * 1.05:
            unrealized_gain = (max_during / exit_price - 1) * 100
            print(f"  ⚠️  We held through peak but exited {unrealized_gain:.1f}% lower!")
        
        if missed_gain > 10:
            print(f"  ❌ MISSED {missed_gain:.1f}% gain after exit!")
        elif missed_gain < -10:
            print(f"  ✅ Avoided {abs(missed_gain):.1f}% drawdown (good exit!)")
        else:
            print(f"  ✓ Exit timing reasonable ({missed_gain:+.1f}% after)")
    
    print("\n" + "="*100)
else:
    print("No trades executed in this period.")

## 4. Position Exposure Timeline: When Are We IN vs OUT?

In [ ]:
# Calculate position exposure
position = pd.Series(0, index=df.index)
in_position = False

for date in df.index:
    if entries[date] and not in_position:
        in_position = True
    if exits[date] and in_position:
        in_position = False
    position[date] = 1 if in_position else 0

# Calculate metrics
days_invested = position.sum()
days_cash = len(position) - days_invested
pct_invested = days_invested / len(position) * 100

print(f"\nMarket Exposure:")
print(f"  Days invested: {days_invested} ({pct_invested:.1f}%)")
print(f"  Days in cash: {days_cash} ({100-pct_invested:.1f}%)")

# Calculate returns while invested vs while in cash
df['returns'] = df['price'].pct_change()
returns_while_invested = df[position == 1]['returns'].sum() * 100
returns_while_cash = df[position == 0]['returns'].sum() * 100

print(f"\nPrice Movement Analysis:")
print(f"  BTC return while we were INVESTED: {returns_while_invested:+.1f}%")
print(f"  BTC return while we were in CASH: {returns_while_cash:+.1f}%")

if returns_while_cash > 20:
    print(f"  ❌ CRITICAL: We missed {returns_while_cash:.1f}% sitting in cash!")
elif returns_while_cash < -20:
    print(f"  ✅ GOOD: We avoided {abs(returns_while_cash):.1f}% crash by being in cash!")
else:
    print(f"  ✓ Neutral: Cash periods had minimal impact")

## 5. Visual: Are We Missing the Rally?

In [ ]:
# Plot price with position exposure
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Top: Price with in/out zones
ax1 = axes[0]
ax1.plot(df.index, df['price'], color='black', linewidth=2, label='BTC Price', alpha=0.8)

# Shade invested periods
invested_periods = position == 1
ax1.fill_between(df.index, 0, df['price'].max() * 1.1, 
                  where=invested_periods, alpha=0.2, color='green', label='Invested')

# Mark entry/exit points
entry_dates = df.index[entries]
exit_dates = df.index[exits]
ax1.scatter(entry_dates, df.loc[entry_dates, 'price'], 
            color='green', marker='^', s=200, label='BUY', zorder=5)
ax1.scatter(exit_dates, df.loc[exit_dates, 'price'], 
            color='red', marker='v', s=200, label='SELL', zorder=5)

# Mark actual tops
ax1.scatter(top_dates, top_prices, color='orange', marker='*', s=300, 
            label='Local Tops', zorder=4, edgecolors='black', linewidths=1)

ax1.set_ylabel('BTC Price ($)', fontsize=12)
ax1.set_title('Signal Timing Analysis: Are We Missing the Tops?', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Bottom: Position size
ax2 = axes[1]
ax2.fill_between(df.index, 0, position, alpha=0.5, color='green', label='Position Size')
ax2.plot(df.index, position, color='darkgreen', linewidth=2)
ax2.set_ylabel('Position (0=Cash, 1=Invested)', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylim(-0.1, 1.1)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Check the chart above:")
print("  • Green zones = We're invested")
print("  • White zones = We're in cash (MISSING gains)")
print("  • Orange stars = Actual local tops")
print("  • Red triangles = Our exit signals")
print("\nQuestion: Do red triangles come BEFORE or AFTER orange stars?")

## 6. Exit Signal Lag Analysis

In [ ]:
# For each exit, find the nearest top
print("\n" + "="*80)
print("EXIT SIGNAL LAG ANALYSIS")
print("="*80)

for exit_date in exit_dates:
    exit_price = df.loc[exit_date, 'price']
    
    # Find nearest top before exit
    tops_before = top_dates[top_dates <= exit_date]
    if len(tops_before) > 0:
        nearest_top_before = tops_before[-1]
        top_price = df.loc[nearest_top_before, 'price']
        days_after_top = (exit_date - nearest_top_before).days
        price_decline = (exit_price / top_price - 1) * 100
        
        print(f"\nExit: {exit_date.date()} @ ${exit_price:,.0f}")
        print(f"  Nearest top: {nearest_top_before.date()} @ ${top_price:,.0f}")
        print(f"  Lag: {days_after_top} days")
        print(f"  Price decline from top: {price_decline:.1f}%")
        
        if days_after_top < 7:
            print(f"  ✅ Exit near the top (< 7 days lag)")
        elif days_after_top < 30:
            print(f"  ⚠️  Exit {days_after_top} days after top")
        else:
            print(f"  ❌ Exit VERY LATE ({days_after_top} days after top)")
        
        if price_decline < -10:
            print(f"  ❌ Exited {abs(price_decline):.1f}% below the top (held through dump!)")
        elif price_decline > -5:
            print(f"  ✅ Exited close to top (only {abs(price_decline):.1f}% lower)")

print("\n" + "="*80)

## 7. Conclusion: Are We Missing the Tops?

In [ ]:
print("\n" + "="*80)
print("VERDICT: ARE WE MISSING THE TOPS?")
print("="*80)

# Calculate opportunity cost
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100
strategy_return = pf.total_return() * 100
opportunity_cost = bh_return - strategy_return

print(f"\n1. PERFORMANCE GAP:")
print(f"   Buy & Hold: {bh_return:.1f}%")
print(f"   Strategy: {strategy_return:.1f}%")
print(f"   Opportunity Cost: {opportunity_cost:.1f}%")

print(f"\n2. TIME IN MARKET:")
print(f"   Invested: {pct_invested:.1f}% of time")
print(f"   In Cash: {100-pct_invested:.1f}% of time")

print(f"\n3. RETURNS BREAKDOWN:")
print(f"   BTC gained {returns_while_invested:+.1f}% while we were INVESTED")
print(f"   BTC gained {returns_while_cash:+.1f}% while we were in CASH")

print(f"\n4. ROOT CAUSE:")
if returns_while_cash > opportunity_cost * 0.5:
    print(f"   ❌ PRIMARY PROBLEM: We sat in CASH during a {returns_while_cash:.1f}% rally!")
    print(f"   ❌ This accounts for most of the {opportunity_cost:.1f}% underperformance")
    print(f"\n   DIAGNOSIS: Exit signals fire TOO EARLY or re-entry signals too weak")
elif abs(returns_while_invested) < bh_return * 0.3:
    print(f"   ❌ PRIMARY PROBLEM: We missed the big moves while invested")
    print(f"   ❌ Only captured {returns_while_invested:.1f}% of {bh_return:.1f}% total")
    print(f"\n   DIAGNOSIS: Exit signals fire TOO LATE (hold through dumps)")
else:
    print(f"   ⚠️  Mixed signals - need deeper investigation")

print(f"\n5. RECOMMENDATION:")
if returns_while_cash > 30:
    print(f"   • Consider: Don't exit completely (use 50/50 hybrid)")
    print(f"   • OR: Weaken exit triggers (require more confirmation)")
    print(f"   • OR: Strengthen re-entry signals (get back in faster)")
elif opportunity_cost > 100:
    print(f"   • Framework fundamentally flawed for this period")
    print(f"   • Missing {opportunity_cost:.0f}% is unacceptable")
    print(f"   • Recommend: Abandon or radically redesign")
else:
    print(f"   • Framework shows promise but needs refinement")
    print(f"   • Test on full history (including bear markets)")

print("\n" + "="*80)